# Trajectory-aligned conditional diffusion (Colab)

Trains the **mechanism-aligned** objective: noise prediction (DDPM) **+** trajectory-consistency loss mapping diffusion times to PDE times.

**Evaluation below uses only final-field agreement** on the held-out `test.pt` split (same metric as the standard notebook for fair comparison).

**GPU:** select a runtime with an **A100** (or any CUDA GPU). **Data:** clone or sync this repo under Google Drive and set `REPO` in the next section.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Point to the repo on Drive

Edit `REPO` to the folder that contains `src/`, `configs/`, and `outputs/`.

In [ ]:
import os
import sys

# --- EDIT THIS PATH ---
REPO = "/content/drive/MyDrive/path/to/heat_operator_recovery"
# ----------------------

os.chdir(REPO)
sys.path.insert(0, REPO)
print("CWD:", os.getcwd())

## 3. Dependencies

In [ ]:
!pip install -q torch numpy matplotlib tqdm pyyaml

## 4. GPU check (expect A100 in Colab Pro+)

In [ ]:
import torch
assert torch.cuda.is_available(), "Enable GPU: Runtime → Change runtime type → GPU"
print(torch.cuda.get_device_name(0))
print("CUDA:", torch.version.cuda)

## 5. Training data

Ensure `outputs/data/train.pt` exists (run `scripts/generate_dataset.py` locally or in Colab first). Config: `configs/default.yaml`.

## 6. Train (trajectory-aligned loss)

Checkpoint: `outputs/checkpoints/model_trajectory_aligned.pt`

In [ ]:
import subprocess
import sys

# subprocess preserves working directory (unlike `!python` in Colab)
subprocess.run(
    [
        sys.executable,
        "-m",
        "src.training.train",
        "--config",
        "configs/default.yaml",
        "--output",
        "model_trajectory_aligned.pt",
    ],
    cwd=REPO,
    check=True,
)

## 7. Evaluate — **final field only** (test split)

Mean squared error between Tweedie-estimated \( \hat{x}_0 \) and ground-truth **last PDE frame** \(u(\cdot, T)\), with random diffusion timestep per forward (same protocol for both notebooks).

In [ ]:
from pathlib import Path

import torch

from src.models.diffusion import GaussianDiffusion
from src.models.unet import SmallUNet
from src.training.eval_metrics import dataset_mean_final_mse
from src.utils.config import load_config

root = Path(REPO)
cfg = load_config(root / "configs" / "default.yaml")
device = torch.device("cuda")

ckpt_path = root / "outputs" / "checkpoints" / "model_trajectory_aligned.pt"
ckpt = torch.load(ckpt_path, map_location=device)
num_diff = int(ckpt["diffusion"]["num_timesteps"])

diffusion = GaussianDiffusion(num_timesteps=num_diff).to(device)
model = SmallUNet(in_ch=2, base=32, time_dim=128).to(device)
model.load_state_dict(ckpt["model"])

test_pt = root / "outputs" / "data" / "test.pt"
bs = int(cfg.get("batch_size", 16))
mse = dataset_mean_final_mse(model, diffusion, test_pt, batch_size=bs, device=device)
print("Trajectory-aligned model — test final-state MSE:", mse)